# Places2 Dataset Overview & Exploration

Introduction to the Places2 scene dataset used as the primary benchmark in deep image inpainting.

## Dataset Overview & Schema

- **Source Used for Analysis:** MIT CSAIL server: [http://data.csail.mit.edu/places/places365/val_large.tar](http://data.csail.mit.edu/places/places365/val_large.tar) and category lists [http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar](http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar)
- **Where to Get Full Dataset Later:** MIT Places2 download page: [http://places2.cs.ailab.mit.edu/download.html](http://places2.cs.ailab.mit.edu/download.html)
- **Full Dataset Size:**
  - Places365-Standard High-Resolution: ~120 GB (~105 GB train + ~14 GB val, 1.8M images)
  - Places365-Standard Small ($256 \times 256$): ~26 GB (~24 GB train + ~2.1 GB val)
  - Full Places2 Challenge (10M images): ~400 GB+
- **Use in Project:** Standard background scene benchmark for inpainting models (LaMa, Telea)

### Standard Places2 Schema
| Field | Type | Description |
| :--- | :--- | :--- |
| `image` | `PIL.Image` | High-resolution scene photograph (RGB, indoor/outdoor). |
| `label` | `int` / `str` | Scene category identifier (e.g. kitchen, street, bedroom). |

> **Inpainting Note:** In inpainting pipelines, Places2 images serve as ground-truth complete scenes. Artificial masks (brush strokes or object masks from PIPE_Mask) are placed over images to train and evaluate background reconstruction.


### 1. Download and extract Places365 validation set and category lists


In [ ]:
!mkdir -p /content/places365
!wget -c http://data.csail.mit.edu/places/places365/val_large.tar -P /content/places365
!ls -lh /content/places365/val_large.tar

--2026-11-22 16:20:40--  http://data.csail.mit.edu/places/places365/val_large.tar
Resolving data.csail.mit.edu (data.csail.mit.edu)... 128.52.131.233
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://data.csail.mit.edu/places/places365/val_large.tar [following]
--2026-11-22 16:20:40--  https://data.csail.mit.edu/places/places365/val_large.tar
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2270320640 (2.1G) [application/x-tar]
Saving to: ‘/content/places365/val_large.tar’

val_large.tar       100%[===================>]   2.11G  53.5MB/s    in 94s     

2026-11-22 16:22:15 (23.0 MB/s) - ‘/content/places365/val_large.tar’ saved [2270320640/2270320640]

-rw-r--r-- 1 root root 2.2G May 16  2016 /content/places365/val_large.tar


In [ ]:
!mkdir -p /content/places365/val_large
!tar -xf /content/places365/val_large.tar -C /content/places365/val_large
!ls /content/places365/val_large | head -n 10

val_large


In [ ]:
!mkdir -p /content/places365/filelists
!wget -c http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar -P /content/places365/filelists
!tar -xf /content/places365/filelists/filelist_places365-standard.tar -C /content/places365/filelists
!ls -la /content/places365/filelists

--2026-11-22 16:39:27--  http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar
Resolving data.csail.mit.edu (data.csail.mit.edu)... 128.52.131.233
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://data.csail.mit.edu/places/places365/filelist_places365-standard.tar [following]
--2026-11-22 16:39:27--  https://data.csail.mit.edu/places/places365/filelist_places365-standard.tar
Connecting to data.csail.mit.edu (data.csail.mit.edu)|128.52.131.233|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67498496 (64M) [application/x-tar]
Saving to: ‘/content/places365/filelists/filelist_places365-standard.tar’

filelist_places365- 100%[===================>]  64.37M  56.4MB/s    in 1.1s    

2026-11-22 16:39:29 (56.4 MB/s) - ‘/content/places365/filelists/filelist_places365-standard.tar’ saved [67498496/67498496]

total 131848
drwxr-xr-x 2 

In [ ]:
import gc
from pathlib import Path
from PIL import Image

img_dir = Path("/content/places365/val_large")
image_paths = sorted(
    p for p in img_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print("Total images found:", len(image_paths))


Total images found: 36500


In [ ]:
from pathlib import Path


def load_places365_val_labels(cat_file: str, val_file: str) -> dict[str, tuple[int, str]]:
  categories = []
  with open(cat_file, "r", encoding="utf-8") as f:
    for line in f:
      parts = line.strip().split()
      if parts:
        categories.append(parts[0])

  fname_to_label = {}
  with open(val_file, "r", encoding="utf-8") as f:
    for line in f:
      parts = line.strip().split()
      if len(parts) >= 2:
        fname, label_id = parts[0], int(parts[1])
        label_name = (
            categories[label_id] if 0 <= label_id < len(categories) else str(label_id)
        )
        fname_to_label[fname] = (label_id, label_name)
  return fname_to_label


# Example once categories_places365.txt and places365_val.txt are in /content/places365
val_map = load_places365_val_labels(
    "/content/places365/categories_places365.txt",
    "/content/places365/filelists/places365_val.txt",
)
for p in image_paths[:5]:
    print(p.name, "->", val_map.get(p.name))

Places365_val_00000001.jpg -> (165, '/g/greenhouse/indoor')
Places365_val_00000002.jpg -> (358, '/w/wet_bar')
Places365_val_00000003.jpg -> (93, '/c/clean_room')
Places365_val_00000004.jpg -> (164, '/g/golf_course')
Places365_val_00000005.jpg -> (289, '/r/rock_arch')


In [ ]:
!cat /content/places365/filelists/categories_places365.txt | head -n 10

/a/airfield 0
/a/airplane_cabin 1
/a/airport_terminal 2
/a/alcove 3
/a/alley 4
/a/amphitheater 5
/a/amusement_arcade 6
/a/amusement_park 7
/a/apartment_building/outdoor 8
/a/aquarium 9


In [ ]:
!cat /content/places365/filelists/places365_val.txt | head -n 10

Places365_val_00000001.jpg 165
Places365_val_00000002.jpg 358
Places365_val_00000003.jpg 93
Places365_val_00000004.jpg 164
Places365_val_00000005.jpg 289
Places365_val_00000006.jpg 106
Places365_val_00000007.jpg 81
Places365_val_00000008.jpg 121
Places365_val_00000009.jpg 150
Places365_val_00000010.jpg 302
